In [1]:
"""
Classes:
  0 = rectangle
  1 = triangle
  2 = circle

"""

import os
import random
import math
import shutil
from pathlib import Path
from PIL import Image, ImageDraw

# ── Config ────────────────────────────────────────────────────────────────────
IMG_W, IMG_H   = 640, 640
NUM_IMAGES     = 1000          # total images to generate
SHAPES_PER_IMG = (1, 5)        # min / max shapes per image
MIN_SIZE       = 40            # smallest shape bounding-box side (px)
MAX_SIZE       = 180           # largest  shape bounding-box side (px)
BG_COLOR       = (255, 255, 255)

SPLIT = {"train": 0.70, "val": 0.20, "test": 0.10}

OUTPUT_DIR = Path("shape_dataset")
CLASSES    = ["rectangle", "triangle", "circle"]

SHAPE_COLORS = [
    (220,  50,  50),   # red
    ( 50, 130, 220),   # blue
    ( 50, 200,  80),   # green
    (230, 150,  40),   # orange
    (150,  60, 200),   # purple
    ( 40, 190, 190),   # teal
]
# ──────────────────────────────────────────────────────────────────────────────


def draw_rectangle(draw, x1, y1, x2, y2, color):
    draw.rectangle([x1, y1, x2, y2], fill=color, outline=(0, 0, 0), width=2)


def draw_circle(draw, x1, y1, x2, y2, color):
    draw.ellipse([x1, y1, x2, y2], fill=color, outline=(0, 0, 0), width=2)


def draw_triangle(draw, x1, y1, x2, y2, color):
    cx  = (x1 + x2) / 2
    pts = [(cx, y1), (x1, y2), (x2, y2)]
    draw.polygon(pts, fill=color, outline=(0, 0, 0))

#Convert pixel bbox → YOLO (cx cy w h) normalised.
# def yolo_label(cls_id, x1, y1, x2, y2, W, H):
#     cx = ((x1 + x2) / 2) / W
#     cy = ((y1 + y2) / 2) / H
#     w  = (x2 - x1) / W
#     h  = (y2 - y1) / H
#     return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def make_image():
    img  = Image.new("RGB", (IMG_W, IMG_H), BG_COLOR)
    draw = ImageDraw.Draw(img)
    labels = []

    n = random.randint(*SHAPES_PER_IMG)
    for _ in range(n):
        size  = random.randint(MIN_SIZE, MAX_SIZE)
        #Random position — bounding box stays fully inside the image
        x1 = random.randint(0, IMG_W - size)
        y1 = random.randint(0, IMG_H - size)
        x2 = x1 + size
        y2 = y1 + size

        color    = random.choice(SHAPE_COLORS)
        cls_id   = random.randint(0, 2)          # 0=rect/1=tri/2=circle

        if cls_id == 0:
            draw_rectangle(draw, x1, y1, x2, y2, color)
        elif cls_id == 1:
            draw_triangle(draw, x1, y1, x2, y2, color)
        else:
            draw_circle(draw, x1, y1, x2, y2, color)

        labels.append(yolo_label(cls_id, x1, y1, x2, y2, IMG_W, IMG_H))

    return img, labels


def write_yaml():
    yaml_path = OUTPUT_DIR / "dataset.yaml"
    abs_path  = OUTPUT_DIR.resolve()
    content   = f"""# Auto-generated YOLO dataset config
path: {abs_path}
train: images/train
val:   images/val
test:  images/test

nc: {len(CLASSES)}
names: {CLASSES}
"""
    yaml_path.write_text(content)
    print(f"  YAML → {yaml_path}")


def main():
    # ── Build folder structure ────────────────────────────────────────────────
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    for split in SPLIT:
        (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    # ── Assign indices to splits ──────────────────────────────────────────────
    indices = list(range(NUM_IMAGES))
    random.shuffle(indices)

    n_train = int(NUM_IMAGES * SPLIT["train"])
    n_val   = int(NUM_IMAGES * SPLIT["val"])

    assignments = {}
    for i, idx in enumerate(indices):
        if i < n_train:
            assignments[idx] = "train"
        elif i < n_train + n_val:
            assignments[idx] = "val"
        else:
            assignments[idx] = "test"

    # ── Generate images ───────────────────────────────────────────────────────
    counts = {"train": 0, "val": 0, "test": 0}
    for idx in range(NUM_IMAGES):
        split = assignments[idx]
        img, labels = make_image()

        stem     = f"img_{idx:05d}"
        img_path = OUTPUT_DIR / "images" / split / f"{stem}.jpg"
        lbl_path = OUTPUT_DIR / "labels" / split / f"{stem}.txt"

        img.save(img_path, "JPEG", quality=95)
        lbl_path.write_text("\n".join(labels))
        counts[split] += 1

        if (idx + 1) % 100 == 0:
            print(f"  {idx + 1}/{NUM_IMAGES} images generated …")

    write_yaml()

    print("\n Dataset ready!")
    print(f"   Train : {counts['train']} images")
    print(f"   Val   : {counts['val']}   images")
    print(f"   Test  : {counts['test']}  images")
    print(f"   Output: {OUTPUT_DIR.resolve()}/")


if __name__ == "__main__":
    random.seed(42)
    main()


  100/1000 images generated …
  200/1000 images generated …
  300/1000 images generated …
  400/1000 images generated …
  500/1000 images generated …
  600/1000 images generated …
  700/1000 images generated …
  800/1000 images generated …
  900/1000 images generated …
  1000/1000 images generated …
  YAML → shape_dataset/dataset.yaml

✅ Dataset ready!
   Train : 700 images
   Val   : 200   images
   Test  : 100  images
   Output: /Users/Rai/shape_dataset/
